In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from pmdarima import ARIMA, auto_arima
from src.common.utils import get_root_directory, collate_array_elements, make_directory
from src.common.stats import adf_test, pp_test
import matplotlib.pyplot as plt
import numpy as np
from src.preprocessing.data_loader import DataLoader
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import mlflow

In [ ]:
# PARAMETERS TO CHANGE
split_size = 0.15
start_date = "01/01/2021"
end_date = "31/12/2023"
maxiter=500
n_periods=7
model_type = "ARIMA"
split = "validation"
experiment_name = f"{model_type}_train_{split}"
max_p = 1
max_q = 7
d = 1
test_p = 1
test_q = 1

In [2]:
root_dir = get_root_directory()
DL = DataLoader(root_dir)
DL.load_data()
DL.set_time_range(start_date=start_date, end_date=end_date)
if split=="validation":
    train, test = DL.split_data(split_size=split_size)
    train, val = DL.split_data(split_type="train_val",split_size=split_size)
    test = test["ETH_D_AvgPrc"]
    train = train["ETH_D_AvgPrc"]
    val = val["ETH_D_AvgPrc"]
    split = val
elif split=="test"
    train, test = DL.split_data(split_size=0.15)
    test = test["ETH_D_AvgPrc"]
    train = train["ETH_D_AvgPrc"]
    split = test

In [21]:
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)

for p in range(1,max_p+1):
    for q in range(1,max_q+1):
        predictions = []
        train_recursive = list(train)
        order = (p, d, q)
        run_name = f"order_({p},{d},{q})"
        with mlflow.start_run(run_name=run_name):
            model = ARIMA(maxiter=maxiter, order=order)
            mlflow.log_params({"p": p,"d": d,"q": q,"maxiter":maxiter,"n_periods": n_periods, "prediction_start_date":split.index.to_list()[0].strftime('%d-%m-%Y')})
            for t in range(len(split)):
                model.fit(train_recursive)
                forecast = model.predict(n_periods=n_periods)
                predictions.append(forecast.tolist())
                train_recursive.append(split.iloc[t])
            df = pd.DataFrame(predictions, columns=['t1','t2','t3','t4','t5','t6','t7'])
            results_file = run_name
            path = str(os.path.join(root_dir,"mlruns",model_type))
            df.to_csv(str(os.path.join(path,results_file)), index=False)
        mlflow.end_run()
        print(f"Completed order: {order}")

Completed order: (1, 1, 1)


In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)

predictions = []
train_recursive = list(train)
order = (test_p, d, test_q)
run_name = f"order_({test_p},{d},{test_q})"
with mlflow.start_run(run_name=run_name):
    model = ARIMA(maxiter=maxiter, order=order)
    mlflow.log_params({"p": test_p,"d": d,"q": test_q,"maxiter":maxiter,"n_periods": n_periods, "prediction_start_date":split.index.to_list()[0].strftime('%d-%m-%Y')})
    for t in range(len(split)):
        model.fit(train_recursive)
        forecast = model.predict(n_periods=n_periods)
        predictions.append(forecast.tolist())
        train_recursive.append(split.iloc[t])
    df = pd.DataFrame(predictions, columns=['t1','t2','t3','t4','t5','t6','t7'])
    results_file = run_name
    path = str(os.path.join(root_dir,"mlruns",model_type))
    df.to_csv(str(os.path.join(path,results_file)), index=False)
    # try:
    #     joblib.dump(model, f"{root_dir}\\mlruns\\ARIMA\\model.pkl")
    #     mlflow.log_artifact(f"{root_dir}\\mlruns\\ARIMA\\model.pkl")
    # except Exception as e:
    #     make_directory(f"{root_dir}\\mlruns\\ARIMA")
    #     joblib.dump(model, f"{root_dir}\\mlruns\\ARIMA\\model.pkl")
    #     mlflow.log_artifact(f"{root_dir}\\mlruns\\ARIMA\\model.pkl")
mlflow.end_run()
print(f"Completed order: {order}")